# Metodología SEMMA – Dataset Pediátrico 2024
**Predicción de Riesgo Cardiovascular en Pacientes Pediátricos**

Metodología empleada: SEMMA (Sample → Explore → Modify → Model → Assess)


In [ ]:
# Librerías globales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             average_precision_score, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight
import joblib

RANDOM_STATE = 42
sns.set_style('whitegrid')
print('Librerías cargadas correctamente.')


---
## 🔹 FASE 1 – SAMPLE


In [ ]:
# Carga del dataset (robusto: CSV o Excel en una sola columna) y normalización de nombres
import os

CANDIDATE_FILES = [
    "dataset_timbiqui_22_por_ciento.csv",   # dataset con 22% de clase positiva
    "datos_pediatricos_timbi_2024.csv",     # dataset base (csv)
    "dataset_timbiqui.csv",                 # alternativa
    "datos pedatricos 2024.xlsx",           # excel (puede venir en una sola columna)
]

data_path = None
for p in CANDIDATE_FILES:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("No se encontró el dataset. Subir uno de estos archivos: " + ", ".join(CANDIDATE_FILES))

# Cargar según extensión
if data_path.lower().endswith(".csv"):
    df_raw = pd.read_csv(data_path)
else:
    df_raw = pd.read_excel(data_path)

print(f"✅ Archivo cargado: {data_path}")
print(f"Dimensiones (raw): {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
print("Columnas (raw):", df_raw.columns.tolist())

# Si el Excel viene en una sola columna cuyo NOMBRE contiene comas (header embebido), reconstruyo
if df_raw.shape[1] == 1 and "," in str(df_raw.columns[0]):
    col_unica = df_raw.columns[0]
    cols = [c.strip() for c in str(col_unica).split(",")]
    df_raw = df_raw[col_unica].astype(str).str.split(",", expand=True)
    df_raw.columns = cols
    print("✅ Dataset reconstruido desde una sola columna.")
    print(f"Dimensiones (reconstruido): {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")

# Normalizar nombres (minúsculas, sin espacios extremos)
df_raw.columns = (
    df_raw.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Renombrar a nombres estándar del estudio
rename_map = {
    "edad": "edad",
    "edadd": "edad",
    "age": "edad",
    "genero": "genero",
    "_genero": "genero",
    "sexo": "genero",
    "gender": "genero",
    "peso_kg": "peso_kg",
    "peso": "peso_kg",
    "pa_sistolica": "pa_sistolica",
    "pasistolica": "pa_sistolica",
    "presion_sistolica": "pa_sistolica",
    "frecuencia_cardiaca": "frecuencia_cardiaca",
    "frecuencia_cardiaca_": "frecuencia_cardiaca",
    "frecuencia_cardiaca__": "frecuencia_cardiaca",
    "frecuencia__cardiaca": "frecuencia_cardiaca",
    "frecuencia_cardiaca": "frecuencia_cardiaca",
    "colesterol_mgdl": "colesterol_mgdl",
    "colesterol": "colesterol_mgdl",
    "riesgo": "riesgo_cv",
    "riesgo_cv": "riesgo_cv",
    "riesgo__cv": "riesgo_cv",
    "riesgo__": "riesgo_cv",
}

df_raw = df_raw.rename(columns={c: rename_map.get(c, c) for c in df_raw.columns})

# Validar columnas mínimas requeridas
required = ["edad","genero","peso_kg","pa_sistolica","frecuencia_cardiaca","colesterol_mgdl","riesgo_cv"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}. Columnas actuales: {df_raw.columns.tolist()}")

df_raw.head()


In [ ]:
# Distribución de la variable objetivo
conteo = df_raw['riesgo_cv'].value_counts()
pct    = df_raw['riesgo_cv'].value_counts(normalize=True) * 100

print('Distribución de clases (riesgo_cv):')
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct.round(2)}))
print(f'\nRatio desbalance: {conteo[0]/conteo[1]:.1f}:1  (Sin riesgo : Con riesgo)')


In [ ]:
# División estratificada train/test (80/20)
X_temp = df_raw.drop(columns=['riesgo_cv'])
y_temp = df_raw['riesgo_cv']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print(f'Conjunto de entrenamiento: {X_train_raw.shape[0]} registros')
print(f'Conjunto de prueba:        {X_test_raw.shape[0]} registros')
print(f'\nProporción riesgo_cv en train:\n{y_train.value_counts(normalize=True).round(4)}')
print(f'\nProporción riesgo_cv en test:\n{y_test.value_counts(normalize=True).round(4)}')


---
## 🔹 FASE 2 – EXPLORE
> Realizo el análisis exploratorio **únicamente sobre el conjunto de entrenamiento** para evitar *data leakage* (fuga de información del test).


In [ ]:
# Unir X_train_raw con y_train para análisis exploratorio
df_train = X_train_raw.copy()
df_train['riesgo_cv'] = y_train.values
df_train.head()


In [ ]:
# Estadísticos descriptivos de variables numéricas
num_cols_raw = ['edad', 'peso_kg', 'frecuencia_cardiaca', 'colesterol_mgdl']
df_train[num_cols_raw].describe().round(2)


In [ ]:
# Distribución de clases (gráfico)
fig, ax = plt.subplots(figsize=(5, 4))
conteo_train = df_train['riesgo_cv'].value_counts()
ax.bar(['Sin riesgo (0)', 'Con riesgo (1)'], conteo_train.values,
       color=['steelblue', 'tomato'], edgecolor='white')
for i, v in enumerate(conteo_train.values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
ax.set_title('Distribución de la Variable Objetivo – Train')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()


In [ ]:
# Histogramas de variables numéricas
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, num_cols_raw):
    df_train[col].dropna().hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')
fig.suptitle('Distribución de Variables Numéricas (Train)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Promedio por clase
print('Promedio de variables numéricas según riesgo_cv:')
df_train.groupby('riesgo_cv')[num_cols_raw].mean().round(2)


In [ ]:
# Nota: pa_sistolica se procesará en MODIFY para poder incluirla en la correlación
# Correlación entre variables numéricas disponibles
corr = df_train[num_cols_raw + ['riesgo_cv']].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlación (Train)')
plt.tight_layout()
plt.show()


---
## 🔹 FASE 3 – MODIFY


In [ ]:
# ── Funciones de limpieza (estandarización) ──────────────────────────────────

def limpiar_genero(val):
    """Estandarizo la columna genero a valores 0 (femenino) y 1 (masculino)."""
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()

    # masculino
    if v in ("masculino", "m", "masc", "male", "h", "hombre"):
        return 1
    # femenino
    if v in ("femenino", "f", "fem", "female", "mujer"):
        return 0

    # si llega como 0/1
    try:
        iv = int(float(v))
        if iv in (0, 1):
            # en este estudio: 1=masculino, 0=femenino
            return iv
    except:
        pass

    return np.nan


def limpiar_pa(val):
    """Convierto pa_sistolica con coma decimal (ej. '109,0') a float."""
    if pd.isna(val):
        return np.nan
    v = str(val).strip().replace(",", ".")
    try:
        return float(v)
    except:
        return np.nan


def preparar_df(df_in, medianas_train=None):
    """Normalizo columnas, limpio variables y aplico imputación usando medianas de train (sin fuga)."""
    df = df_in.copy()

    # Asegurar nombres estándar (por si cambia el archivo)
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # Renombrar posibles variantes
    df = df.rename(columns={
        "pa_sistolica": "pa_sistolica",
        "presion_sistolica": "pa_sistolica",
        "frecuencia_cardiaca": "frecuencia_cardiaca",
        "colesterol_mgdl": "colesterol_mgdl",
        "riesgo": "riesgo_cv",
        "riesgo_cv": "riesgo_cv",
    })

    # Limpiezas específicas
    if "genero" in df.columns:
        df["genero"] = df["genero"].apply(limpiar_genero)

    if "pa_sistolica" in df.columns:
        df["pa_sistolica"] = df["pa_sistolica"].apply(limpiar_pa)

    # Convertir numéricos
    for col in ["edad", "peso_kg", "frecuencia_cardiaca", "colesterol_mgdl", "pa_sistolica", "genero"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Imputación (medianas de train)
    cols_imputar = ["peso_kg", "colesterol_mgdl", "pa_sistolica", "genero"]
    if medianas_train is None:
        medianas_train = {c: df[c].median() for c in cols_imputar if c in df.columns}

    for c in cols_imputar:
        if c in df.columns and c in medianas_train:
            df[c] = df[c].fillna(medianas_train[c])

    return df, medianas_train


print("Funciones de limpieza definidas.")


In [ ]:
# Limpieza e imputación SIN fuga de información:
# 1) Calculo medianas en train
df_train_clean, medianas_train = preparar_df(X_train_raw, medianas_train=None)

# 2) Aplico las mismas medianas al test
df_test_clean, _ = preparar_df(X_test_raw, medianas_train=medianas_train)

print("Nulos en train tras limpieza:")
print(df_train_clean.isnull().sum())
print("\nNulos en test tras limpieza:")
print(df_test_clean.isnull().sum())


In [ ]:
# Features finales (nombres estándar)
FEATURES = ["edad", "genero", "peso_kg", "pa_sistolica", "frecuencia_cardiaca", "colesterol_mgdl"]

X_train = df_train_clean[FEATURES].copy()
X_test  = df_test_clean[FEATURES].copy()

print(f"Features utilizadas: {FEATURES}")
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")


In [ ]:
# Escalado (ajuste sobre train, transformación sobre ambos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Escalado aplicado (StandardScaler).')

# Class weights por desbalance
clases = np.unique(y_train)
pesos  = compute_class_weight('balanced', classes=clases, y=y_train)
class_weight_dict = dict(zip(clases, pesos))
print(f'Class weights: {class_weight_dict}')


---
## 🔹 FASE 4 – MODEL


In [ ]:
# Definición de modelos
# Nota: GradientBoostingClassifier no acepta class_weight;
# se compensa con sample_weight en el fit.
modelos = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', max_depth=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', n_estimators=200, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        random_state=RANDOM_STATE)
}

# Sample weights para Gradient Boosting (equivalente a class_weight='balanced')
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

resultados = {}   # almacenará métricas por modelo
modelos_fit = {}  # almacenará modelos entrenados

print('Iniciando entrenamiento...')
for nombre, modelo in modelos.items():
    if nombre == 'Logistic Regression':
        # Requiere escalado
        modelo.fit(X_train_scaled, y_train)
        y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
        y_pred = modelo.predict(X_test_scaled)
    elif nombre == 'Gradient Boosting':
        # No acepta class_weight; usa sample_weight
        modelo.fit(X_train, y_train, sample_weight=sample_weights)
        y_prob = modelo.predict_proba(X_test)[:, 1]
        y_pred = modelo.predict(X_test)
    else:
        modelo.fit(X_train, y_train)
        y_prob = modelo.predict_proba(X_test)[:, 1]
        y_pred = modelo.predict(X_test)

    modelos_fit[nombre] = modelo
    resultados[nombre] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'pr_auc':  average_precision_score(y_test, y_prob)
    }
    print(f'  ✔ {nombre} entrenado.')

print('\nEntrenamiento completado.')


In [ ]:
# Resumen de métricas por modelo
filas = []
for nombre, res in resultados.items():
    rep = classification_report(y_test, res['y_pred'], output_dict=True)
    filas.append({
        'Modelo':      nombre,
        'Accuracy':    round(rep['accuracy'], 4),
        'Precision_1': round(rep['1']['precision'], 4),
        'Recall_1':    round(rep['1']['recall'], 4),
        'F1_1':        round(rep['1']['f1-score'], 4),
        'ROC-AUC':     round(res['roc_auc'], 4),
        'PR-AUC':      round(res['pr_auc'], 4)
    })

df_metricas = pd.DataFrame(filas).set_index('Modelo')
print('Métricas de evaluación (conjunto de prueba):')
df_metricas


---
## 🔹 FASE 5 – ASSESS


In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (nombre, res) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Sin riesgo', 'Con riesgo'],
                yticklabels=['Sin riesgo', 'Con riesgo'], ax=ax)
    ax.set_title(nombre)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')
fig.suptitle('Matrices de Confusión', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Curvas ROC
fig, ax = plt.subplots(figsize=(7, 5))
colores = {'Logistic Regression': 'steelblue',
           'Decision Tree': 'darkorange',
           'Random Forest': 'seagreen',
           'Gradient Boosting': 'purple'}
for nombre, res in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{nombre} (AUC={res['roc_auc']:.3f})",
            color=colores[nombre], linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curvas ROC – Comparación de Modelos')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Curvas Precision-Recall
fig, ax = plt.subplots(figsize=(7, 5))
for nombre, res in resultados.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ax.plot(rec, prec, label=f"{nombre} (PR-AUC={res['pr_auc']:.3f})",
            color=colores[nombre], linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall – Comparación de Modelos')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Comparación final de modelos
print('═'*60)
print('COMPARACIÓN FINAL DE MODELOS')
print('═'*60)
print(df_metricas.to_string())

mejor_modelo_nombre = df_metricas['ROC-AUC'].idxmax()
mejor_roc = df_metricas.loc[mejor_modelo_nombre, 'ROC-AUC']
mejor_pr  = df_metricas.loc[mejor_modelo_nombre, 'PR-AUC']
mejor_f1  = df_metricas.loc[mejor_modelo_nombre, 'F1_1']

print(f'\n✔ MEJOR MODELO: {mejor_modelo_nombre}')
print(f'  ROC-AUC : {mejor_roc}')
print(f'  PR-AUC  : {mejor_pr}')
print(f'  F1 (c1) : {mejor_f1}')
print('═'*60)


In [ ]:
# Selección y justificación del modelo final
print(f"""
SELECCIÓN DEL MODELO FINAL
══════════════════════════
Modelo seleccionado : {mejor_modelo_nombre}

Justificación:
  El modelo {mejor_modelo_nombre} obtuvo el mayor ROC-AUC ({mejor_roc:.4f})
  y PR-AUC ({mejor_pr:.4f}), métricas que son prioritarias en contextos
  de desbalance de clases. Adicionalmente, alcanzó el mejor F1-score para
  la clase positiva (Con riesgo = 1) con un valor de {mejor_f1:.4f},
  lo que indica un equilibrio adecuado entre Precision y Recall.
  En aplicaciones clínicas pediátricas, minimizar los falsos negativos
  (pacientes con riesgo no detectados) es crítico, lo que refuerza
  la elección basada en estas métricas.
""")


In [ ]:
# Guardado del modelo final
modelo_final = modelos_fit[mejor_modelo_nombre]
joblib.dump(modelo_final, 'modelo_final_riesgo_pediatrico.joblib')
print(f'Modelo guardado como: modelo_final_riesgo_pediatrico.joblib')

# Guardar también el scaler (necesario para Logistic Regression en producción)
joblib.dump(scaler, 'scaler_riesgo_pediatrico.joblib')
print('Scaler guardado como: scaler_riesgo_pediatrico.joblib')
